# Notebook 02 — Data Extraction and Preprocessing

This notebook extracts and preprocesses satellite data from Google Earth Engine for Afghanistan's 399 districts (2000–2025).

| Variable | Source | Native Resolution | Temporal | Aggregation |
|----------|--------|-------------------|----------|-------------|
| NDVI | MODIS MOD13A3 | 1 km | Monthly | Mean |
| LST | MODIS MOD11A2 | 1 km | 8-day | Monthly mean |
| Rainfall | CHIRPS Daily | 5.5 km | Daily | Monthly total |

Processing pipeline:
1. Quality masking to remove cloud/snow-affected pixels
2. Monthly temporal composites
3. Spatial aggregation to district-level means
4. Export to CSV and GeoJSON

Outputs:

`afg_climate_2000_2025.csv`

`afg_district_lookup.geojson`

---
## Setup and Data Sources

Load Earth Engine image collections and Afghanistan district boundaries from FAO GAUL Level 2.

| Collection | Description | Key Band |
|------------|-------------|----------|
| `MODIS/061/MOD13A3` | Monthly vegetation indices | NDVI |
| `MODIS/061/MOD11A2` | 8-day land surface temperature | LST_Day_1km |
| `UCSB-CHG/CHIRPS/DAILY` | Daily rainfall estimates | precipitation |

In [1]:
import ee
import pandas as pd
from datetime import datetime

# Initialize Earth Engine
ee.Initialize()

In [2]:
# Load Earth Engine image collections
ndvi_ic = ee.ImageCollection("MODIS/061/MOD13A3")           # monthly NDVI
lst_ic  = ee.ImageCollection("MODIS/061/MOD11A2")           # 8-day LST
chirps_ic = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")     # daily rainfall

# Load boundaries and filter for Afghanistan districts
gaul2 = ee.FeatureCollection("FAO/GAUL/2015/level2")   # district boundaries
af_districts = gaul2.filter(ee.Filter.eq("ADM0_NAME", "Afghanistan"))

## Define Study Period

Determine data availability across all three sources and generate monthly date intervals.

- **Start date:** March 2000 (earliest common availability for MODIS products)
- **End date:** November 2025 (CHIRPS data lag)
- **Total months:** ~309 monthly observations per district

In [3]:
def start_date(ic):
    return ee.Date(
        ic.aggregate_min('system:time_start')
    ).format('YYYY-MM-dd').getInfo()

print("Earliest data availability:")
print("NDVI: ", start_date(ndvi_ic))
print("LST:  ", start_date(lst_ic))
print("Rain: ", start_date(chirps_ic))

Earliest data availability:
NDVI:  2000-02-01
LST:   2000-02-18
Rain:  1981-01-01


In [4]:
# generate monthly date intervals for 2000–2025
t0 = "2000-03-01"
t1 = "2025-11-30" # CHIRPS has data only up to Nov 2025

def month_range(start, end):
    dates = pd.date_range(start=start, end=end, freq="MS")
    return [(d.strftime("%Y-%m-%d"),
             (d + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d"))
            for d in dates]

months = month_range(t0, t1)
date_list = ee.List(months) # convert to an Earth Engine–compatible list

print(f"Months: {len(months)}")

Months: 309


## Quality Masking Functions

Apply quality assurance masks to remove unreliable pixels before aggregation.

| Product | QA Band | Mask Criteria |
|---------|---------|---------------|
| **NDVI** | SummaryQA | Keep values ≤ 1 (good/marginal quality) |
| **LST** | QC_Day | Keep where bits 0–1 = 0 (good quality) |

CHIRPS rainfall requires no quality masking (already gap-filled product).

In [5]:
#  NDVI QA mask (MOD13A3)
def mask_ndvi(img):
    summary = img.select('SummaryQA')
    mask = summary.lte(1)
    return img.updateMask(mask)


# LST QA mask (MOD11A2)
def mask_lst(img):
    qc = img.select('QC_Day')
    mask = qc.bitwiseAnd(3).eq(0)
    return img.updateMask(mask)

## Monthly Composite Processing

Create monthly composite images by aggregating each variable with appropriate methods.

| Variable | Aggregation | Scale Factor | Unit Conversion |
|----------|-------------|--------------|-----------------|
| **NDVI** | Mean | × 0.0001 | Raw DN to index (−1 to 1) |
| **LST** | Mean | × 0.02, − 273.15 | Kelvin to Celsius |
| **Rainfall** | Sum | None | mm/month |

In [6]:
# Create monthly composite image (mean NDVI, mean LST, total rainfall) 
def process_month(m_dates):
    start = ee.List(m_dates).get(0)
    end   = ee.List(m_dates).get(1)
    date  = ee.Date(start)

    ndvi = (ndvi_ic.filterDate(start, end)
            .map(mask_ndvi) # mask
            .select('NDVI')
            .mean()
            .multiply(0.0001)) # MODIS scale factor
   
    lst = (lst_ic.filterDate(start, end)
           .map(mask_lst) # mask
           .select('LST_Day_1km')
           .mean()
           .multiply(0.02) # MODIS scale factor
           .subtract(273.15) # convert to Celsius
           .rename('LST'))
   
    rain = (chirps_ic.filterDate(start, end)
            .select('precipitation')
            .sum()
            .rename('RAIN'))

    img = ee.Image.cat([ndvi, lst, rain])

# tag with time metadata
    return img.set({
        'date': start,
        'system:time_start': date.millis()
    })

In [7]:
# Band-specific native scales 
ndvi_scale = ndvi_ic.first().select('NDVI').projection().nominalScale()
lst_scale  = lst_ic.first().select('LST_Day_1km').projection().nominalScale()
rain_scale = chirps_ic.first().select('precipitation').projection().nominalScale()

## Spatial Aggregation to Districts

Reduce raster composites to district-level statistics using `reduceRegions()`.

Considerations:
- Each band reduced at its native resolution to preserve accuracy
- Results joined on `ADM2_CODE` (unique district identifier)
- Geometry stripped from output to minimize file size
- `tileScale=4` used to handle memory limits for large reductions

In [8]:
# Reduce each band separately with its own scale, then join back on ADM2_CODE
def strip_geometry(f):
    """Remove geometry from a feature to keep output small."""
    return ee.Feature(None, f.toDictionary())

def reduce_band(img, band_name, scale, out_prop):
    """Reduce a single band to district averages."""
    date_str = img.get('date')
    reducer = ee.Reducer.mean().setOutputs([out_prop])
    fc = img.select(band_name).reduceRegions(
        collection=af_districts,
        reducer=reducer,
        scale=scale,
        tileScale=4
    )
    return fc.map(lambda f: strip_geometry(f).set('date', date_str))

def join_on_adm2(left_fc, right_fc, right_prop):
    """Merge FeatureCollections using ADM2_CODE as the key."""
    join = ee.Join.saveFirst('match')
    filt = ee.Filter.equals(leftField='ADM2_CODE', rightField='ADM2_CODE')
    joined = ee.FeatureCollection(join.apply(left_fc, right_fc, filt))

    def merge(f):
        m = f.get('match')
        val = ee.Algorithms.If(m, ee.Feature(m).get(right_prop), None)
        merged = ee.Feature(f).set(right_prop, val)
        keep = merged.propertyNames().remove('match')
        return merged.select(keep)

    return joined.map(merge)

def reduce_collection(img):
    """Reduce all bands for one month and join results."""
    fc_ndvi = reduce_band(img, 'NDVI', ndvi_scale, 'NDVI')
    fc_lst  = reduce_band(img, 'LST',  lst_scale,  'LST')
    fc_rain = reduce_band(img, 'RAIN', rain_scale, 'RAIN')

    fc = join_on_adm2(fc_ndvi, fc_lst, 'LST')
    fc = join_on_adm2(fc, fc_rain, 'RAIN')
    return fc

In [9]:
# Defines the columns included in the time series export
export_columns = ['ADM2_CODE', 'ADM2_NAME', 'ADM1_NAME', 'date', 'NDVI', 'LST', 'RAIN']

# Build the time series FeatureCollection (geometry stripped) and select columns
final_fc = ee.FeatureCollection(
    date_list.map(lambda m: reduce_collection(process_month(m)))).flatten()

final_fc = final_fc.select(propertySelectors=export_columns, retainGeometry=False)

print(f"Time series columns: {export_columns}")

Time series columns: ['ADM2_CODE', 'ADM2_NAME', 'ADM1_NAME', 'date', 'NDVI', 'LST', 'RAIN']


---
## Export to Google Drive

Export two files for downstream analysis:

| File | Format | Contents |
|------|--------|----------|
| `afg_climate_2000_2025.csv` | CSV | Monthly NDVI, LST, rainfall per district |
| `afg_district_lookup.geojson` | GeoJSON | District boundaries with ADM codes |

In [10]:
# EXPORT 1: Time Series (no geometry)
export_timeseries = ee.batch.Export.table.toDrive(
    collection=final_fc,
    description="afg_climate_2000_2025",
    fileFormat="CSV"
)
export_timeseries.start()
print(f"Export started\nColumns:", export_columns)

Export started
Columns: ['ADM2_CODE', 'ADM2_NAME', 'ADM1_NAME', 'date', 'NDVI', 'LST', 'RAIN']


In [11]:
# EXPORT 2: District Lookup GeoJSON
lookup_cols = ['ADM2_CODE', 'ADM2_NAME', 'ADM1_NAME']

# Select only the lookup columns, keeping geometry
district_lookup = af_districts.select(propertySelectors=lookup_cols, retainGeometry=True)

export_lookup = ee.batch.Export.table.toDrive(
    collection=district_lookup,
    description="afg_district_lookup",
    fileFormat="GeoJSON"
)
export_lookup.start()
print("Export started\nColumns:", lookup_cols, "+ geometry")

Export started
Columns: ['ADM2_CODE', 'ADM2_NAME', 'ADM1_NAME'] + geometry


In [12]:
# Check status export tasks
tasks = ee.batch.Task.list()
for task in tasks[:2]:
    status = task.status()
    print(f"{status['description']}: {status['state']}")
    print(task.status())

afg_district_lookup: READY
{'state': 'READY', 'description': 'afg_district_lookup', 'priority': 100, 'creation_timestamp_ms': 1768261569240, 'update_timestamp_ms': 1768261569240, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'HKVQP6PDLAEFCGXTVRPLLGSS', 'name': 'projects/ee-af-air-quality/operations/HKVQP6PDLAEFCGXTVRPLLGSS'}
afg_climate_2000_2025: READY
{'state': 'READY', 'description': 'afg_climate_2000_2025', 'priority': 100, 'creation_timestamp_ms': 1768261568970, 'update_timestamp_ms': 1768261568970, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'YZRKBBLMJ5CFB6OORRXW2H2H', 'name': 'projects/ee-af-air-quality/operations/YZRKBBLMJ5CFB6OORRXW2H2H'}
